# How to train it: the strategy steps A–D

The completed training study, in the order it was worked out. **Step A:** two-phase training (contrastive pre-training, then the classifier) with the main loss. **Step B:** the same with the second loss. **Step C:** phase 2 done properly — freeze the encoder except the last layer and fine-tune. **Step D:** all-in-one-phase training, with the contrastive loss as a regulariser, and a variant trained on hard samples. The notebook closes with the first version's training cycles for comparison and a divergence check between models.

**Where this fits:** this section tests a different defence idea — train the detector with **supervised contrastive learning** so its feature space separates the classes more cleanly, then attack the result with the machinery of `03-attacks/`. The CW machinery aimed at these models continues the lineage of `03-attacks/cw_sine_lineage.ipynb`.

*Consolidated from `malware_detection_contrast_learning_compeleted.ipynb` (the spine), `malware_detection_contrast_learning.ipynb` and `malware_detection_contrast_learning2.ipynb` (originally in `contrastive-learning`). Outputs cleared.*

## Step 1 — Setup and the shared pieces

# Load dataset and libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import time
import os
import math
from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd
import tqdm
import pickle

torch.manual_seed(0)
np.random.seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/model_DNN_drebin_best.pth',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/best_model_AT_imax.pth',
]

output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
from adverserial_attacks_functions import *

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array, dtype=torch.uint8).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array, dtype=torch.uint8).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

# SupContrast Model (Encoder + Projector)

In [ ]:
def extract_features(model, dataloader):
    model.cpu()
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            output, z = model.CoreModel(inputs.to(torch.float32), return_z=True)
            features.append(z.cpu().numpy())
            labels.append(targets.cpu().numpy())

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)

    return features, labels

def extract_features2(model, dataloader):
    model.cpu()
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            z = model.get_intermediate_output(inputs.to(torch.float32))
            features.append(z.cpu().numpy())
            labels.append(targets.cpu().numpy())

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)

    return features, labels



def apply_tsne(features):
    tsne = TSNE(n_components=2, random_state=42)
    tsne_results = tsne.fit_transform(features)
    return tsne_results

def plot_tsne_and_density(tsne_results, labels):
    df = pd.DataFrame({
        'tsne_1': tsne_results[:, 0],
        'tsne_2': tsne_results[:, 1],
        'label': labels.squeeze()
    })

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 14))

    # t-SNE scatter plot
    sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, ax=ax1, legend='full')
    ax1.set_title('t-SNE Scatter Plot')

    # Density plot with transparency and contours
    sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, fill=True, alpha=0.70, ax=ax2)
    ax2.set_title('Density Plot')

    plt.show()

In [ ]:
import torch
import torch.nn.functional as F

def calculate_divergences(z, labels):
    # Ensure the tensors are on the same device
    device = z.device
    labels = labels.to(device)

    # Compute cosine similarities
    cos_sim_z = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)

    # Convert cosine similarities to cosine distances
    cos_distances_z = 1 - cos_sim_z

    # Initialize variables
    N = len(z)
    intra_class_distances = []
    inter_class_distances = []

    for i in range(N):
        # Identify positive and negative sets for the anchor
        positive_indices = (labels == labels[i]).nonzero(as_tuple=True)[0]
        negative_indices = (labels != labels[i]).nonzero(as_tuple=True)[0]

        # Compute intra-class divergence for benign and adversarial examples
        intra_class_distance_z = cos_distances_z[i][positive_indices].mean().item()

        # Compute inter-class divergence for benign and adversarial examples
        inter_class_distance_z = cos_distances_z[i][negative_indices].mean().item()

        intra_class_distances.append(intra_class_distance_z)
        inter_class_distances.append(inter_class_distance_z)

    # Calculate absolute intra-class and inter-class divergences
    d_a_plus = torch.tensor(intra_class_distances).mean().item()
    d_a_minus = torch.tensor(inter_class_distances).mean().item()

    # Calculate relative intra-class divergence (R-DIV)
    d_r_plus = d_a_plus / d_a_minus

    print("Absolute Intra-Class Divergence:", d_a_plus)
    print("Absolute Inter-Class Divergence:", d_a_minus)
    print("Relative Intra-Class Divergence (R-DIV):", d_r_plus)


    return d_a_plus, d_a_minus, d_r_plus

# Example usage
batch_size = 100
latent_dim = 200
num_classes = 2

# Generate random latent representations and labels for demonstration
z = torch.rand(batch_size, latent_dim)
labels = torch.randint(0, num_classes, (batch_size,))

# Calculate divergences
d_a_plus, d_a_minus, d_r_plus = calculate_divergences(z, labels)

In [ ]:
import torch
from torch._C import device
import torch.nn.functional as F

def distance(x, y, dist, pairwise=False):
    """
        Pairwise distance
        Args:
            x, y: input pair
            dist: distance type ["l2", "l1", "linf", "cosine"]
            pairwise:
        Note:
            Need renormalize if using l1
    """

    assert(len(x.shape) == 2)
    b, d = x.shape

    if pairwise:
        x = torch.unsqueeze(x, dim=0) # [1, b, d]
        y = torch.unsqueeze(y, dim=1) # [b, 1, d]

    if dist == 'l2':
        loss = torch.sqrt(torch.sum(torch.square(x - y), dim=-1, keepdim=False)) # [b,b]

    elif dist == 'l1':
        loss = torch.sum(torch.abs(x - y), dim=-1, keepdim=False) # [b,b]

    elif dist == 'linf':
        loss, _ = torch.max(torch.abs(x - y), dim=-1, keepdim=False) # [b,b]

    elif dist == 'lkinf':
        k = 10
        t, _ = torch.topk(torch.abs(x - y), k=k, dim=-1)
        loss = torch.sum(t, dim=-1, keepdim=False)
        loss /= k

    elif dist == 'cosine':
        # Note that: cosine_similarity range [-1,1]
        # dist=-1 <--> very similar
        # dist=0 <--> orthogonal
        # dist=1 <--> very disimilar
        loss = 0 - torch.cosine_similarity(x, y, dim=-1)

    elif dist == 'matmul':
        x = torch.squeeze(x) # [b,d]
        y = torch.squeeze(y) # [b,d]
        loss = - torch.matmul(x, y.T)

    # loss = loss / torch.max(loss).detach() # normalizing to avoid numerical problem

    if pairwise:
        assert(loss.shape[0] == b)
        assert(loss.shape[1] == b)
    else:
        assert(loss.shape[0] == x.shape[0]) # [b, ]

    return loss

def get_positive_mask(labels):
    # ATTENTION HERE, positive mask will ignore the diagonal

    # l = torch.matmul(labels, labels.T) # [2b,2b]
    # m = torch.ones_like(l).fill_diagonal_(0) # [2b,2b]
    # pos_mask = torch.multiply(l, m) # [2b,2b]

    l = torch.matmul(labels, labels.T).fill_diagonal_(0) # [2b,2b]
    return l

def get_negative_mask(labels):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    return inv_mask

def get_hard_negative_mask(labels, preds):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    hard_mask = torch.matmul(labels, preds.T) # [2b, 2b], mask for tile(y_j) == y_i
    neg_mask = torch.multiply(inv_mask, hard_mask) # [2b, 2b], mask for (y_j!= y_i & tile(y_j) == y_i)
    return neg_mask


def get_soft_negative_mask(labels, preds):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    soft_mask = torch.matmul(preds, preds.T) # [2b, 2b], mask for tile(y_j) == tile(y_i)
    neg_mask = torch.multiply(inv_mask, soft_mask) # [2b, 2b], mask for (y_j!= y_i & tile(y_j) == y_i)
    return neg_mask

LARGE_NUM = 1e9

def soft_lcscl(hidden,
             labels,
             preds,
             dist,
             neg_type,
             hidden_norm=True,
             temperature=0.07):
    """Compute Local Supervised Contrastive Loss.
    Args:
        hidden: hidden vector (`Tensor`) of shape (2 * bsz, dim).
        labels: labels of hidden vector of shape (2*bsz, classes)
        preds: current prediction
        hidden_norm: whether or not to use normalization on the hidden vector.
        temperature: a `floating` number for temperature scaling.
        dist: distance metric in latent space {'l1', 'l2', 'cosine', 'linf'}
        neg_type:
            'hard': negative set when y_j ~= y_i, but tilde(y_j) == y_i
            'soft': negative set when y_j ~= y_i, but tilde(y_j) == tilde(y_i)
            'merge': negative set when y_i ~= y_i but tilde(y_j) in [y_i,  tilde(y_i)]
            'leaking':
    Returns:
        A loss scalar.

    the choice of temperature:
        temperature = 0.5 based on simCLR paper
        temperature = 0.07 based on SCL paper (they claim that smaller temperature
        benefit training more than higher ones)
    """
    #if dist in ['matmul', 'l1', 'l2']:
        #assert(hidden_norm is True)
    #elif dist in ['cosine']:
        #assert(hidden_norm is False)

    # Get (normalized) hidden1 and hidden2.
    #if hidden_norm:
        # gradient will pass through the normalization step
        #hidden = F.normalize(hidden, p=2, dim=-1)


    batch_size = hidden.shape[0]

    # Remove illegal elements
    same_preds = get_positive_mask(preds)
    same_labels = get_positive_mask(labels)
    diff_labels = get_negative_mask(labels)

    same_preds_same_labels = torch.multiply(same_preds, same_labels)
    same_preds_diff_labels = torch.multiply(same_preds, diff_labels)

    pos_mask = get_positive_mask(labels)
    # neg_mask = tf.ones_like(pos_mask) - pos_mask # INCORRECT, because it consider the diagonal as NEGATIVE
    neg_mask = get_negative_mask(labels) # CORRECT
    hard_neg_mask = get_hard_negative_mask(labels, preds)
    soft_neg_mask = get_soft_negative_mask(labels, preds)
    merge_mask = torch.minimum(hard_neg_mask+soft_neg_mask, torch.ones_like(hard_neg_mask))

    # ATTENTION HERE, NEED VERIFIED BY EXPERIMENT, Stop_gradient make it worse
    # pos_mask = pos_mask.detach()
    # neg_mask = neg_mask.detach()
    # hard_neg_mask = hard_neg_mask.detach()
    # soft_neg_mask = soft_neg_mask.detach()
    # merge_mask = merge_mask.detach()
    # same_preds_same_labels = same_preds_same_labels.detach()
    #
    if neg_type == 'all':
        illegal_mask = torch.zeros_like(neg_mask)
        nb_neg = torch.sum(neg_mask, dim=-1)

    elif neg_type == 'hard':
        illegal_mask = torch.maximum(neg_mask - hard_neg_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(hard_neg_mask, dim=-1)

    elif neg_type == 'soft':
        illegal_mask = torch.maximum(neg_mask - soft_neg_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(soft_neg_mask, dim=-1)

    elif neg_type == 'merge':
        illegal_mask = torch.maximum(neg_mask - merge_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(merge_mask, dim=-1)

    elif neg_type == 'leaking':
        illegal_mask = torch.maximum(torch.ones_like(same_preds_same_labels) - same_preds_same_labels - same_preds_diff_labels, torch.zeros_like(same_preds_same_labels))
        #print(illegal_mask)
        nb_neg = torch.sum(same_preds_diff_labels, dim=-1)

    # Pairwise Similarity
    masks = F.one_hot(torch.arange(batch_size), num_classes=batch_size).to(hidden.device)
    sim = - distance(hidden, hidden, dist, pairwise=True) / temperature
    logits_12 = sim - masks * LARGE_NUM
    logits_masked = logits_12 - LARGE_NUM * illegal_mask
    logprobs = F.log_softmax(logits_masked, dim=1)

    """ Convert labels to pairwise labels of shape [2b, 2b], diagonal(labels)=0
        multiple hot label vectors

    """
    # Cross entropy loss with multiple hot labels

    if neg_type == 'leaking':
        loss = - torch.sum(same_preds_same_labels * logprobs, dim=-1) # [2b, ]
        nb_pos = torch.maximum(torch.sum(same_preds_same_labels, dim=-1), torch.ones_like(loss)) # [2b, ]
    else:
        loss = - torch.sum(pos_mask * logprobs, dim=-1) # [2b, ]
        nb_pos = torch.maximum(torch.sum(pos_mask, dim=-1), torch.ones_like(loss)) # [2b, ]

    # if True:
    #     scale = torch.div(nb_neg, nb_neg+nb_pos) # V1
    #     # scale = torch.div(nb_neg, nb_pos) #V3
    #     # scale = nb_neg #V2
    #     loss = torch.multiply(loss, scale)

    assert(loss.shape == nb_pos.shape)
    #print('nb_pos',nb_pos)
    #print('nb_neg',nb_neg)
    loss = torch.div(loss, nb_pos) # [2b,]
    loss = torch.mean(loss, dim=0) # [1,]

    return loss

In [ ]:
class SupConLoss(nn.Module):
    """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
    It also supports the unsupervised contrastive loss in SimCLR"""
    def __init__(self, temperature=0.07, contrast_mode='all',
                 base_temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """Compute loss for model. If both `labels` and `mask` are None,
        it degenerates to SimCLR unsupervised loss:
        https://arxiv.org/pdf/2002.05709.pdf

        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        """
        device = (torch.device('cuda')
                  if features.is_cuda
                  else torch.device('cpu'))

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        # modified to handle edge cases when there is no positive pair
        # for an anchor point.
        # Edge case e.g.:-
        # features of shape: [4,1,...]
        # labels:            [0,1,1,2]
        # loss before mean:  [nan, ..., ..., nan]
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        # loss
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss

In [ ]:
# Define the Linear Classifier(NO NEED)
class LinearClassifier(nn.Module):
    def __init__(self, dim_in=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(dim_in, num_classes)

    def forward(self, x):
        return self.fc(x)


# Define the CoreModel
class CoreModel(nn.Module):
    def __init__(self):
        super(CoreModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(200, 200)
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x, return_z=False):
        x = F.relu(self.fc1(x))
        #x = self.dropout1(x)
        z = F.relu(self.fc2(x))
        z2 = self.dropout2(z)
        output  = self.fc3(z2)
        if return_z:
            return output, z

        return output

# Define the Projection Head
class ProjHead(nn.Module):
    def __init__(self, dim_in=200, feat_dim=128):
        super(ProjHead, self).__init__()
        self.head = nn.Linear(dim_in, feat_dim)

    def forward(self, x):
        #x = F.normalize(x, dim=1)
        z_proj = self.head(x)
        z_proj = F.normalize(z_proj, dim=1)
        return z_proj



# Define the Wrapper
class Wrapper(nn.Module):
    def __init__(self, CoreModel, ProjLayer):
        super(Wrapper, self).__init__()
        self.CoreModel = CoreModel
        self.ProjLayer = ProjLayer
        #self.LC = LC

    def forward(self, x, return_z=False):
        output, z = self.CoreModel(x, return_z=True)
        z_proj = self.ProjLayer(z)
        #output = self.LC(z)
        if return_z:
            return output, z_proj
        else:
            return output

# Instantiate the components
core_model = CoreModel()
proj_head = ProjHead(200, 128)
#linear_classifier = LinearClassifier(128, 2)

# Combine them in the Wrapper
combined_model = Wrapper(core_model, proj_head)

# Example usage
x = torch.randn(2, 10000)  # Example input
output, z_proj = combined_model(x,return_z=True)
print(output.shape)
print(z_proj.shape)

## Step A — Two-phase, main loss, no dropout

# Step: A

two phase

main loss

no dropout

In [ ]:
def training(model, train_loader, val_loader, epochs=100, lr=0.001, weight_decay=0., device='cuda', verbose=True):
    # Define parameters for the loss function
    dist = 'cosine'
    hidden_norm = False
    temperature = 0.07
    neg_type='all'

    criterion = SupConLoss(temperature=temperature)

    # Define Loss Function and Optimizer
    #class_weights = torch.tensor([0.11, 0.89]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr,weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.1, anneal_strategy='cos', cycle_momentum=False)

    best_val_loss = torch.inf
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        total_samples = 0

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            bsz = x_batch.shape[0]

            if bsz < 100:
                continue

            optimizer.zero_grad()

            # Forward pass
            outputs, z_proj = model(x_batch, return_z=True)

            # Calculate loss
            loss_train = criterion(z_proj.unsqueeze(1), y_batch)
            #print(loss_train0)

            # Calculate loss
            #predicted = outputs.argmax(1)
            #labels_one_hot = F.one_hot(y_batch.long(), num_classes=2).squeeze(1).float()
            #predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
            #loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
            #loss_train = loss_scl
            #print(loss_train)
            #print('**********')

            # Backward pass and optimization
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            # Update total loss and samples count
            epoch_loss += loss_train.item() * bsz
            total_samples += bsz

        # Calculate average loss for the epoch
        mean_loss = epoch_loss / total_samples
        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f}')

        model.eval()
        epoch_loss = 0
        total_samples = 0
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
                bsz = x_val.shape[0]

                outputs, z_proj = model(x_val, return_z=True)

                # Calculate loss1
                loss = criterion(z_proj.unsqueeze(1), y_val)

                # Calculate loss2
                #predicted = outputs.argmax(1)
                #labels_one_hot = F.one_hot(y_val.long(), num_classes=2).squeeze(1).float()
                #predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
                #loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
                #epoch_loss += (loss_scl.mean() * samples)

                # Update total loss and samples count
                epoch_loss += loss.item() * bsz
                total_samples += bsz

        # Calculate average loss for the epoch
        val_loss = epoch_loss / total_samples

        if val_loss <= best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_nn_cl.pth')

        if verbose:
            print(f"\tValidation loss: {val_loss:.4f}")
            print(f"\tModel selected at epoch {best_epoch} with validation loss {best_val_loss:.4f}")
            print('******************************************************************************************')

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 14350
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model, loss function, and optimizer
core_model = CoreModel().to(device)
proj_head = ProjHead(200, 128).to(device)

combined_model = Wrapper(core_model, proj_head).to(device)
#combined_model.load_state_dict(torch.load('/content/model_AT_max (14).pth'))

In [ ]:
epochs = 20
training(combined_model.to(device), train_loader, val_loader, epochs, lr=0.02, weight_decay=1e-4, device=device, verbose=True)

In [ ]:
combined_model1 = Wrapper(core_model, proj_head).to(device)
combined_model1.load_state_dict(torch.load('/content/model_nn_cl.pth'))

In [ ]:
features, labels = extract_features(combined_model1, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 1024
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
epochs = 20
training(combined_model.to(device), train_loader, val_loader, epochs, lr=0.005, weight_decay=1e-4, device=device,verbose=True)

In [ ]:
combined_model2 = Wrapper(core_model, proj_head).to(device)
combined_model2.load_state_dict(torch.load('/content/model_nn_cl.pth'))

In [ ]:
features, labels = extract_features(combined_model2, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

## Step B — Two-phase, second loss

# Step: B

two phase

second loss

no dropout

In [ ]:
def training(model, train_loader, val_loader, epochs=100, lr=0.001, weight_decay=0., device='cuda', verbose=True):
    # Define parameters for the loss function
    dist = 'matmul'
    hidden_norm = False
    temperature = 0.07
    neg_type='leaking'

    criterion = SupConLoss(temperature=temperature)

    # Define Loss Function and Optimizer
    #class_weights = torch.tensor([0.11, 0.89]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr,weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.1, anneal_strategy='cos', cycle_momentum=False)

    best_val_loss = torch.inf
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        total_samples = 0

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            bsz = x_batch.shape[0]

            if bsz < 100:
                continue

            optimizer.zero_grad()

            # Forward pass
            outputs, z_proj = model(x_batch, return_z=True)

            # Calculate loss
            #loss_train = criterion(z_proj.unsqueeze(1), y_batch)
            #print(loss_train0)

            # Calculate loss
            predicted = outputs.argmax(1)
            labels_one_hot = F.one_hot(y_batch.long(), num_classes=2).squeeze(1).float()
            predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
            loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
            loss_train = loss_scl
            #print(loss_train)
            #print('**********')

            # Backward pass and optimization
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            # Update total loss and samples count
            epoch_loss += loss_train.item() * bsz
            total_samples += bsz

        # Calculate average loss for the epoch
        mean_loss = epoch_loss / total_samples
        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f}')

        model.eval()
        epoch_loss = 0
        total_samples = 0
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
                bsz = x_val.shape[0]

                outputs, z_proj = model(x_val, return_z=True)

                # Calculate loss1
                #loss = criterion(z_proj.unsqueeze(1), y_val)

                # Calculate loss2
                predicted = outputs.argmax(1)
                labels_one_hot = F.one_hot(y_val.long(), num_classes=2).squeeze(1).float()
                predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
                loss = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)

                # Update total loss and samples count
                epoch_loss += loss.item() * bsz
                total_samples += bsz

        # Calculate average loss for the epoch
        val_loss = epoch_loss / total_samples

        if val_loss <= best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_nn_cl.pth')

        if verbose:
            print(f"\tValidation loss: {val_loss:.4f}")
            print(f"\tModel selected at epoch {best_epoch} with validation loss {best_val_loss:.4f}")
            print('******************************************************************************************')

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 4096
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
epochs = 20
training(combined_model.to(device), train_loader, val_loader, epochs, lr=0.01, weight_decay=1e-4, device=device, verbose=True)

In [ ]:
combined_model3 = Wrapper(core_model, proj_head).to(device)
combined_model3.load_state_dict(torch.load('/content/model_nn_cl.pth'))

In [ ]:
features, labels = extract_features(combined_model3, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 128
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
epochs = 20
training(combined_model.to(device), train_loader, val_loader, epochs, lr=0.0001, weight_decay=1e-4, device=device,verbose=True)

In [ ]:
combined_model4 = Wrapper(core_model, proj_head).to(device)
combined_model4.load_state_dict(torch.load('/content/model_nn_cl.pth'))

In [ ]:
features, labels = extract_features(combined_model4, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

## Step C — Phase 2: fine-tune with the encoder frozen

# Step: C
phase 2

In [ ]:
def training2(model, train_loader, val_loader, epochs=100, lr=0.001, weight_decay=0., device='cuda', verbose=True):

    # Define Loss Function and Optimizer
    class_weights = torch.tensor([0.11, 0.89]).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.05, anneal_strategy='cos', cycle_momentum=False)

    best_f1_val = 0.
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        total_samples = 0
        epoch_accuracy = 0

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            bsz = x_batch.shape[0]

            if bsz < 100:
                continue

            optimizer.zero_grad()

            # Forward pass
            outputs, z_proj = model(x_batch, return_z=True)
            predicted = outputs.argmax(1)

            # Calculate loss
            loss_train = criterion(outputs, y_batch.view(-1).long())

            # Backward pass and optimization
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            # Update total loss and samples count
            epoch_loss += loss_train.item() * bsz
            epoch_accuracy += (predicted == y_batch.view(-1)).sum().item()
            total_samples += bsz

        mean_loss = epoch_loss / total_samples
        mean_accuracy = (epoch_accuracy / total_samples) * 100

        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        model.eval()
        epoch_loss = 0
        total_samples = 0
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
                bsz = x_val.shape[0]
                outputs = model(x_val)
                loss = criterion(outputs, y_val.view(-1).long())

                epoch_loss += loss.item() * bsz
                total_samples += bsz

                predicted = outputs.argmax(1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(y_val.view(-1).cpu().numpy())

        # Calculate average loss for the epoch
        val_loss = epoch_loss / total_samples
        f1_val = f1_score(all_labels, all_preds, average='binary')
        if f1_val >= best_f1_val:
            best_f1_val = f1_val
            best_loss_val = val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_nn.pth')

        if verbose:
            print(f"\tValidation loss: {val_loss:.4f} | Validation F1 Score: {f1_val:.4f}")
            print(f"\tModel selected at epoch {best_epoch} with validation F1 Score {best_f1_val:.4f} and Validation loss: {best_loss_val:.4f}")
            print('******************************************************************************************')

In [ ]:
combined_model2

In [ ]:
for name, param in combined_model2.named_parameters():
  print(name)

In [ ]:
# Freeze all layers in CoreModel except the last layer
for name, param in combined_model2.named_parameters():
    if name != 'CoreModel.fc3.weight' and name != 'CoreModel.fc3.bias':
        param.requires_grad = False
        print(name)

In [ ]:
for name, param in combined_model2.named_parameters():
  print(f'{name}: {"trainable" if param.requires_grad else "frozen"}')

it's not strictly necessary to filter the parameters in the optimizer initialization if you've already set requires_grad to False for the parameters you want to freeze. PyTorch will automatically ignore parameters with requires_grad set to False during optimization. However, filtering the parameters can be a good practice to explicitly ensure that only trainable parameters are included in the optimizer. This can improve readability and reduce potential confusion.



```

# Define the optimizer to include only the trainable parameters
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, combined_model.parameters()), lr=0.001)
```



In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 256
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
training2(combined_model2, train_loader, val_loader, epochs=100, lr=0.0005, weight_decay=1e-4, device=device, verbose=True)

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model, loss function, and optimizer
core_model = CoreModel().to(device)
proj_head = ProjHead(200, 128).to(device)

combined_model = Wrapper(core_model, proj_head).to(device)
combined_model.load_state_dict(torch.load('/content/model_nn.pth'))

In [ ]:
model_evaluation(combined_model.to(device), test_loader, device)

In [ ]:
# Define different attacks with their parameters
attacks = [
    (dfgsm_k, {"k":10, "epsilon":0.02, 'random':True, 'is_report_loss_diff' : False}),
    (bga_k, {"k":10, 'is_report_loss_diff' : False}),
    (bca_k, {"k":10, 'is_report_loss_diff' : False}),
    (grosse_k, {"k":10, 'is_report_loss_diff' : False}),
    (mimicry, {'removal_array':removal_array, "trials":1, 'is_report_loss_diff':False}),
    (mimicry, {'removal_array':removal_array, "trials":10, 'is_report_loss_diff':False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}),
]

# Iterate over each attack and its parameters
for attack_func, attack_params in attacks:
    print(f"Running attack: {attack_func.__name__} with parameters: {attack_params}")
    adv_predict(test_loader, combined_model, attack_func, device, **attack_params)
    print('********************************************')  # Print an empty line for separation

## Step D — One phase, contrastive loss as a regulariser

#Step: D
all in one phase and contrastive loss as a regularization

In [ ]:
def training3(model, train_loader, val_loader, epochs=100, lr=0.001, weight_decay=0., beta=1., device='cuda', verbose=True):
    temperature = 0.07
    criterion_cl = SupConLoss(temperature=temperature)
    # Define Loss Function and Optimizer
    class_weights = torch.tensor([0.11, 0.89]).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.05, anneal_strategy='cos', cycle_momentum=False)

    best_f1_val = 0.
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        total_samples = 0
        epoch_accuracy = 0

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            bsz = x_batch.shape[0]

            if bsz < 100:
                continue

            optimizer.zero_grad()

            # Forward pass
            outputs, z_proj = model(x_batch, return_z=True)
            predicted = outputs.argmax(1)

            # Calculate loss
            loss_scl = criterion_cl(z_proj.unsqueeze(1), y_batch)
            loss_at = criterion(outputs, y_batch.view(-1).long())
            loss_train = loss_at + beta * loss_scl

            # Backward pass and optimization
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            # Update total loss and samples count
            epoch_loss += loss_train.item() * bsz
            epoch_accuracy += (predicted == y_batch.view(-1)).sum().item()
            total_samples += bsz

        mean_loss = epoch_loss / total_samples
        mean_accuracy = (epoch_accuracy / total_samples) * 100

        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        model.eval()
        epoch_loss = 0
        total_samples = 0
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
                bsz = x_val.shape[0]

                # Forward pass
                outputs, z_proj = model(x_val, return_z=True)
                predicted = outputs.argmax(1)

                # Calculate loss
                loss_scl = criterion_cl(z_proj.unsqueeze(1), y_val)
                loss_at = criterion(outputs, y_val.view(-1).long())
                loss = loss_at + beta * loss_scl

                epoch_loss += loss.item() * bsz
                total_samples += bsz

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(y_val.view(-1).cpu().numpy())

        # Calculate average loss for the epoch
        val_loss = epoch_loss / total_samples
        f1_val = f1_score(all_labels, all_preds, average='binary')
        if f1_val >= best_f1_val:
            best_f1_val = f1_val
            best_loss_val = val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_nn.pth')

        if verbose:
            print(f"\tValidation loss: {val_loss:.4f} | Validation F1 Score: {f1_val:.4f}")
            print(f"\tModel selected at epoch {best_epoch} with validation F1 Score {best_f1_val:.4f} and Validation loss: {best_loss_val:.4f}")
            print('******************************************************************************************')

In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model, loss function, and optimizer
core_model = CoreModel().to(device)
proj_head = ProjHead(200, 128).to(device)

combined_model = Wrapper(core_model, proj_head).to(device)

In [ ]:
epochs = 100
training3(combined_model.to(device), train_loader, val_loader, epochs, lr=0.001, weight_decay=1e-4, beta=1., device=device, verbose=True)

## Step D′ — One phase, trained on hard samples

# Step D:

training on one phase with hard samples(based on the second paper)

In [ ]:
def training4(model, train_loader, val_loader, epochs=100, lr=0.001, weight_decay=0., beta=1., device='cuda', verbose=True):

    # Define Loss Function and Optimizer
    dist = 'cosine'
    neg_type = 'leaking'  # You can change this to 'soft', 'merge', or 'leaking'
    hidden_norm = False
    temperature = 0.07

    class_weights = torch.tensor([0.11, 0.89]).to(device)
    criterion_at = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.05, anneal_strategy='cos', cycle_momentum=False)

    best_f1_val = 0.
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        total_samples = 0
        epoch_accuracy = 0

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            bsz = x_batch.shape[0]

            if bsz < 100:
                continue

            optimizer.zero_grad()

            # Forward pass
            outputs, z_proj = model(x_batch, return_z=True)
            predicted = outputs.argmax(1)

            # Calculate loss
            loss_at = criterion_at(outputs, y_batch.view(-1).long())
            labels_one_hot = F.one_hot(y_batch.long(), num_classes=2).squeeze(1).float()
            predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
            loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
            loss_train = loss_at + beta * loss_scl

            # Backward pass and optimization
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            # Update total loss and samples count
            epoch_loss += loss_train.item() * bsz
            epoch_accuracy += (predicted == y_batch.view(-1)).sum().item()
            total_samples += bsz

        mean_loss = epoch_loss / total_samples
        mean_accuracy = (epoch_accuracy / total_samples) * 100

        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        model.eval()
        epoch_loss = 0
        total_samples = 0
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for x_val, y_val in val_loader:
                x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
                bsz = x_val.shape[0]

                # Forward pass
                outputs, z_proj = model(x_val, return_z=True)
                predicted = outputs.argmax(1)

                # Calculate loss
                loss_at = criterion_at(outputs, y_val.view(-1).long())
                labels_one_hot = F.one_hot(y_val.long(), num_classes=2).squeeze(1).float()
                predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
                loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
                loss = loss_at + beta * loss_scl

                epoch_loss += loss.item() * bsz
                total_samples += bsz

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(y_val.view(-1).cpu().numpy())

        # Calculate average loss for the epoch
        val_loss = epoch_loss / total_samples
        f1_val = f1_score(all_labels, all_preds, average='binary')
        if f1_val >= best_f1_val:
            best_f1_val = f1_val
            best_loss_val = val_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_nn.pth')

        if verbose:
            print(f"\tValidation loss: {val_loss:.4f} | Validation F1 Score: {f1_val:.4f}")
            print(f"\tModel selected at epoch {best_epoch} with validation F1 Score {best_f1_val:.4f} and Validation loss: {best_loss_val:.4f}")
            print('******************************************************************************************')

In [ ]:
epochs = 100
training4(combined_model.to(device), train_loader, val_loader, epochs, lr=0.001, weight_decay=1e-4, beta=1., device=device, verbose=True)

In [ ]:
features, labels = extract_features(combined_model, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

## The first version's cycles, and the DNN baseline for comparison


[A Visual Guide to Learning Rate Schedulers in PyTorch](https://towardsdatascience.com/a-visual-guide-to-learning-rate-schedulers-in-pytorch-24bbb262c863)



In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 128
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)

In [ ]:
def validate(model, val_loader, device):
    model.eval()
    correct_val = 0
    n_samples = 0
    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            n_samples += len(x_val)
            outputs = model(x_val)
            predicted = outputs.argmax(1)
            correct_val += (predicted == y_val.view(-1)).sum().item()
    return correct_val / n_samples

def validate(model, val_loader, device):
  model.eval()
  all_preds = []
  all_labels = []
  with torch.no_grad():
      for x_val, y_val in val_loader:
          x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
          outputs = model(x_val)
          predicted = outputs.argmax(1)
          all_preds.extend(predicted.cpu().numpy())
          all_labels.extend(y_val.view(-1).cpu().numpy())
  return f1_score(all_labels, all_preds, average='binary')

def training(model, train_loader, val_loader, epochs=100, lr=0.001, weight_decay=0., beta=1., device='cuda', verbose=True, **kwargs):
    # Define parameters for the loss function
    dist = 'cosine'
    neg_type = 'leaking'
    hidden_norm = False
    temperature = 0.07

    # Define Loss Function and Optimizer
    class_weights = torch.tensor([0.11, 0.89]).to(device)
    criterion_at = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=total_steps, pct_start=0.05, anneal_strategy='cos', cycle_momentum=False)

    best_f1_val = 0.
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        epoch_accuracies = []

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            batch_size = x_batch.shape[0]

            if batch_size < 100:
                continue

            optimizer.zero_grad()

            # Forward pass
            outputs, z_proj = model(x_batch, return_z=True)
            predicted = outputs.argmax(1)

            # Calculate loss
            loss_at = criterion_at(outputs, y_batch.view(-1).long())
            labels_one_hot = F.one_hot(y_batch.long(), num_classes=2).squeeze(1).float()
            predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
            loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
            loss_train = loss_at + beta * loss_scl

            # Backward pass and optimization
            loss_train.backward()
            optimizer.step()
            scheduler.step()

            epoch_losses.append(loss_train.item())
            acc_train = (predicted == y_batch.view(-1)).sum().item() / len(y_batch)
            epoch_accuracies.append(acc_train)


        mean_loss = np.mean(epoch_losses)
        mean_accuracy = np.mean(epoch_accuracies) * 100

        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        f1_val = validate(model, val_loader, device)
        if f1_val >= best_f1_val:
            best_f1_val = f1_val
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_nn_cl.pth')

        if verbose:
            print(f"\tValidation F1 Score: {f1_val:.4f}")
            print(f"\tModel selected at epoch {best_epoch} with validation F1 Score {best_f1_val:.4f}")
            print('******************************************************************************************')

In [ ]:
epochs = 100
training(combined_model.to(device), train_loader, val_loader, epochs, lr=0.001, weight_decay=1e-4, beta=1., device=device, verbose=True)

In [ ]:
model_evaluation(combined_model2.to(device), test_loader, device)

In [ ]:
# Define different attacks with their parameters
attacks = [
    (dfgsm_k, {"k":10, "epsilon":0.02, 'random':True, 'is_report_loss_diff' : False}),
    (bga_k, {"k":10, 'is_report_loss_diff' : False}),
    (bca_k, {"k":10, 'is_report_loss_diff' : False}),
    (grosse_k, {"k":10, 'is_report_loss_diff' : False}),
    (mimicry, {'removal_array':removal_array, "trials":1, 'is_report_loss_diff':False}),
    (mimicry, {'removal_array':removal_array, "trials":10, 'is_report_loss_diff':False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}),
]

# Iterate over each attack and its parameters
for attack_func, attack_params in attacks:
    print(f"Running attack: {attack_func.__name__} with parameters: {attack_params}")
    adv_predict(test_loader, combined_model2, attack_func, device, **attack_params)
    print('********************************************')  # Print an empty line for separation

In [ ]:
model_evaluation(model_DNN.to(device), test_loader, device)

In [ ]:
features, labels = extract_features2(model_DNN, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

In [ ]:
# Define different attacks with their parameters
attacks = [
    (dfgsm_k, {"k":10, "epsilon":0.06, 'random':True, 'is_report_loss_diff' : False}),
    (bga_k, {"k":10, 'is_report_loss_diff' : False}),
    (bca_k, {"k":10, 'is_report_loss_diff' : False}),
    (grosse_k, {"k":10, 'is_report_loss_diff' : False}),
    (mimicry, {'removal_array':removal_array, "trials":1, 'is_report_loss_diff':False}),
    (mimicry, {'removal_array':removal_array, "trials":10, 'is_report_loss_diff':False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 0.06, 'norm': 'linf', 'is_report_loss_diff' : False}),
]

# Iterate over each attack and its parameters
for attack_func, attack_params in attacks:
    print(f"Running attack: {attack_func.__name__} with parameters: {attack_params}")
    adv_predict(test_loader, model_DNN, attack_func, device, **attack_params)
    print('********************************************')  # Print an empty line for separation

## A divergence check between the trained models

#divergence

In [ ]:
import torch
import torch.nn as nn

class MalwareDetectionModel(nn.Module):
    def __init__(self):
        super(MalwareDetectionModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(200, 200)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.6)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        return x

    def get_intermediate_output(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        return x

In [ ]:
model = MalwareDetectionModel().to(device)
# Load model parameters
model.load_state_dict(torch.load('model_DNN_drebin_best.pth', map_location=torch.device(device)))

In [ ]:
# Create an instance of your model
model_AT_imax = MalwareDetectionModel().to(device)

# Load model parameters
model_AT_imax.load_state_dict(torch.load('best_model_AT_imax.pth', map_location=torch.device(device)))

In [ ]:
def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done


def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf', initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    if random:
       round_threshold = torch.rand(x_next.size())
    x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next

In [ ]:
for idx_batch, (x_batch, y_batch) in enumerate(test_loader):
    x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
    batch_size = x_batch.shape[0]

    if batch_size < 100:
        continue

    # Separate malicious and benign samples
    mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
    mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]

    # Generate adversarial examples

    #{'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
    #pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
    with torch.enable_grad():
      pertb_mal_x = pgd_min2(mal_x_batch, mal_y_batch, model, insertion_array, removal_array, k=100, step_length=1., norm='l1', is_report_loss_diff=True)

    x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
    y_batch = torch.cat([y_batch, mal_y_batch])

    z = model.get_intermediate_output(x_batch.to(torch.float32))
    d_a_plus, d_a_minus, d_r_plus = calculate_divergences(z, y_batch)
    break